### 기본 라이브러리

In [1]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()
# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False


# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# 혼동행렬 확인 
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# 피처중요도 확인 
import shap
import matplotlib.pyplot as plt
import pandas as pd
import xgboost as xgb

import os
import joblib

### 계층형

In [4]:
# === 저장된 모델 불러오기 ===
model_dirs = {
    "E_NotE": "./model/E_NotE/LightGBM_gridsearch_model.pkl",
    "D_NotD": "./model/D_NotD/weight/lgbm_class_weight_model.pkl",
    "C_AB": "./model/C_AB/LightGBM_gridsearch_model.pkl",
    "A_B": "./model/A_B/LightGBM_gridsearch_model.pkl"
}

In [5]:
models = {}
features = {}

for key, path in model_dirs.items():
    loaded = joblib.load(path)
    models[key] = loaded["model"].best_estimator_
    features[key] = loaded["features"]

In [6]:
# === 계층형 예측 함수 (E → D → C → A/B) ===
def predict_segment(df_raw: pd.DataFrame) -> np.ndarray:
    df = df_raw.copy()

    # 단계별 피처 준비
    X_e = df[features["E_NotE"]]
    X_d = df[features["D_NotD"]]
    X_c = df[features["C_AB"]]
    X_ab = df[features["A_B"]]

    result = np.full(len(df), 'Unclassified')

    # 1단계: E vs Not E
    pred_e = models["E_NotE"].predict(X_e)
    mask_not_e = (pred_e == 1)
    result[~mask_not_e] = 'E'

    # 2단계: D vs Not D (for Not E only)
    pred_d = models["D_NotD"].predict(X_d[mask_not_e])
    mask_d = (pred_d == 0)
    result[mask_not_e] = 'NotD'  # 일단 전체 NotD로 초기화
    result[mask_not_e & mask_d] = 'D'  # D로 재할당

    # 3단계: C vs A/B (for Not D only)
    mask_not_d = mask_not_e & ~mask_d
    pred_c = models["C_AB"].predict(X_c[mask_not_d])
    mask_ab = (pred_c == 1)
    result[mask_not_d & ~mask_ab] = 'C'

    # 4단계: A vs B (for A/B only)
    mask_ab_final = mask_not_d & mask_ab
    pred_ab = models["A_B"].predict(X_ab[mask_ab_final])
    result[mask_ab_final & (pred_ab == 0)] = 'A'
    result[mask_ab_final & (pred_ab == 1)] = 'B'

    return result

#### 예측

In [7]:
from functools import reduce

In [8]:
import pyarrow.parquet as pq

# 1. 테스트 파일 경로
test_path = "./data/test"
files = [f for f in os.listdir(test_path) if f.endswith(".parquet")]

# 2. 예측에 필요한 전체 피처 리스트 + 병합 기준 컬럼 포함
all_features = set()
for feat_set in features.values():
    all_features |= set(feat_set)
all_features |= {"ID", "기준년월"}

# 3. 필요한 컬럼만 선택적으로 불러오기
dfs = []
for file in files:
    path = os.path.join(test_path, file)
    
    # Parquet 메타데이터에서 컬럼 이름만 가져오기
    parquet_file = pq.ParquetFile(path)
    sample_cols = parquet_file.schema.names

    use_cols = list(all_features & set(sample_cols))

    if "ID" not in use_cols:
        continue  # 병합 불가 파일은 제외

    df = pd.read_parquet(path, columns=use_cols)
    dfs.append(df)

In [9]:
# 기준년월 포함해서 병합
test_df = reduce(lambda left, right: pd.merge(left, right, on=["ID", "기준년월"], how="outer"), dfs)

In [10]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600000 entries, 0 to 599999
Data columns (total 37 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   기준년월             600000 non-null  int64 
 1   이용금액_R3M_신용체크    600000 non-null  int64 
 2   _2순위카드이용금액       600000 non-null  int64 
 3   ID               600000 non-null  object
 4   카드이용한도금액         600000 non-null  int64 
 5   CA한도금액           600000 non-null  int64 
 6   할부금액_3M_R12M     600000 non-null  int64 
 7   이용금액_오프라인_B0M    600000 non-null  int64 
 8   이용금액_오프라인_R6M    600000 non-null  int64 
 9   정상입금원금_B5M       600000 non-null  int64 
 10  정상입금원금_B2M       600000 non-null  int64 
 11  쇼핑_도소매_이용금액      600000 non-null  int64 
 12  이용개월수_결제일_R6M    600000 non-null  int64 
 13  _2순위업종_이용금액      600000 non-null  int64 
 14  _1순위교통업종_이용금액    600000 non-null  int64 
 15  _3순위쇼핑업종_이용금액    600000 non-null  int64 
 16  이용건수_신용_R12M     600000 non-null  int64 
 17  이용건수_오프라인_

In [12]:
def predict_segment(df_raw: pd.DataFrame) -> np.ndarray:
    df = df_raw.copy()

    # 존재하는 컬럼만 추출
    X_e = df[[col for col in features["E_NotE"] if col in df.columns]]
    X_d = df[[col for col in features["D_NotD"] if col in df.columns]]
    X_c = df[[col for col in features["C_AB"] if col in df.columns]]
    X_ab = df[[col for col in features["A_B"] if col in df.columns]]

    # 결과 초기화
    result = np.full(len(df), 'Unclassified', dtype=object)

    # Step 1: E vs Not E
    pred_step1 = models["E_NotE"].predict(X_e)
    mask_not_e = (pred_step1 == 1)
    result[~mask_not_e] = 'E'

    # Step 2: D vs Not D (Only Not E 대상)
    idx_not_e = np.where(mask_not_e)[0]
    pred_step2 = models["D_NotD"].predict(X_d.iloc[idx_not_e])
    mask_d = (pred_step2 == 0)  # D인 경우
    idx_d = idx_not_e[mask_d]
    result[idx_d] = 'D'

    # Step 3: C vs A/B (Only Not E & Not D 대상)
    idx_not_d = idx_not_e[~mask_d]
    pred_step3 = models["C_AB"].predict(X_c.iloc[idx_not_d])
    mask_ab = (pred_step3 == 1)
    
    # C 예측
    idx_c = idx_not_d[~mask_ab]
    result[idx_c] = 'C'

    # Step 4: A vs B (Only A/B 대상)
    idx_ab = idx_not_d[mask_ab]
    pred_step4 = models["A_B"].predict(X_ab.iloc[idx_ab])
    result[idx_ab[pred_step4 == 0]] = 'A'
    result[idx_ab[pred_step4 == 1]] = 'B'

    return result

In [14]:
# 예측 수행
predicted_segment = predict_segment(test_df)

# 제출 파일 저장
submission = pd.DataFrame({
    "ID": test_df["ID"],
    "Segment": predicted_segment
})
submission.to_csv("./submission/submission.csv", index=False)
print("제출 파일 저장 완료: ./submission/submission.csv")

제출 파일 저장 완료: ./submission/submission.csv


In [15]:
print(pd.Series(predicted_segment).value_counts())

C    264243
A    157992
E    124686
D     53079
Name: count, dtype: int64
